In [0]:
silver_df = spark.table("retailmart.silver.website_customers_clean")

display(silver_df)

customer_id,first_name,last_name,email,phone,city,country,registration_date,marketing_opt_in,silver_load_time
WC000001,Fay,Otto,thijn94@example.org,0243254198,Achtmaal,Belgium,2024-06-01,Yes,2026-08-04T10:11:29.133709Z
WC000002,Koen,Meis,joeyhoogers@example.net,0431402327,Neerijnen,Germany,2025-02-05,Yes,2026-08-04T10:11:29.133709Z
WC000003,Isa,Mansvelt,lois95@example.net,0682615699,Sellingen,Belgium,2024-02-14,No,2026-08-04T10:11:29.133709Z
WC000004,Lisanne,Kolen,puk59@example.org,0286192770,Molkwerum,France,2023-10-07,Yes,2026-08-04T10:11:29.133709Z
WC000005,Ali,Evers,jinthe91@example.com,+310097882653,Haghorst,Belgium,2025-04-24,Yes,2026-08-04T10:11:29.133709Z
WC000006,Mohamed,Van Der Maath,van-ghoerlemette@example.com,0354074399,Stad Aan 't Haringvliet,France,2024-06-11,No,2026-08-04T10:11:29.133709Z
WC000007,Lieke,Van Leeuwen,billungzakaria@example.com,+31882012303,Liempde,Belgium,2024-04-29,Yes,2026-08-04T10:11:29.133709Z
WC000008,Sienna,Le Briel,daan64@example.org,0640486746,Finsterwolde,Germany,2026-05-22,No,2026-08-04T10:11:29.133709Z
WC000009,Nathan,Kwaadland,kaylee82@example.com,+31239723486,Eagum,France,2025-07-22,No,2026-08-04T10:11:29.133709Z
WC000010,Marit,Verheij,pslagmolen@example.com,0764005722,Loosdrecht,France,2024-04-22,Yes,2026-08-04T10:11:29.133709Z


In [0]:
from pyspark.sql.functions import *

In [0]:
country_summary = (
    silver_df
    .groupBy("country")
    .agg(
        count("*").alias("total_customers")
    )
    .orderBy(desc("total_customers"))
)

display(country_summary)

country,total_customers
Belgium,1029
Netherlands,1012
France,991
Germany,968


In [0]:
gold_country_path = (
    "abfss://gold@stretailmartdev011.dfs.core.windows.net/"
    "customer_country_summary"
)

(
    country_summary.write
    .format("delta")
    .mode("overwrite")
    .save(gold_country_path)
)

print("Gold country summary written successfully.")

Gold country summary written successfully.


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS retailmart.gold
""")

DataFrame[]

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS retailmart.gold.customer_country_summary
USING DELTA
LOCATION '{gold_country_path}'
""")

DataFrame[]

In [0]:
display(
    spark.table("retailmart.gold.customer_country_summary")
)

country,total_customers
Belgium,1029
Netherlands,1012
France,991
Germany,968


In [0]:
total = silver_df.count()

country_summary = (
    country_summary
    .withColumn(
        "customer_percentage",
        round(
            col("total_customers") * 100 / total,
            2
        )
    )
)

display(country_summary)

country,total_customers,customer_percentage
Belgium,1029,25.73
Netherlands,1012,25.3
France,991,24.78
Germany,968,24.2


In [0]:
(
    country_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_country_path)
)

In [0]:
display(
    spark.table("retailmart.gold.customer_country_summary")
)

country,total_customers,customer_percentage
Belgium,1029,25.73
Netherlands,1012,25.3
France,991,24.78
Germany,968,24.2
